### All days of the challange:

* [Day 1: Handling missing values](https://www.kaggle.com/rtatman/data-cleaning-challenge-handling-missing-values)
* [Day 2: Scaling and normalization](https://www.kaggle.com/rtatman/data-cleaning-challenge-scale-and-normalize-data)
* [Day 3: Parsing dates](https://www.kaggle.com/rtatman/data-cleaning-challenge-parsing-dates/)
* [Day 4: Character encodings](https://www.kaggle.com/rtatman/data-cleaning-challenge-character-encodings/)
* [Day 5: Inconsistent Data Entry](https://www.kaggle.com/rtatman/data-cleaning-challenge-inconsistent-data-entry/)
___
Welcome to day 1 of the 5-Day Data Challenge! Today, we're going to be looking at how to deal with missing values. To get started, click the blue "Fork Notebook" button in the upper, right hand corner. This will create a private copy of this notebook that you can edit and play with. Once you're finished with the exercises, you can choose to make your notebook public to share with others. :)

> **Your turn!** As we work through this notebook, you'll see some notebook cells (a block of either code or text) that has "Your Turn!" written in it. These are exercises for you to do to help cement your understanding of the concepts we're talking about. Once you've written the code to answer a specific question, you can run the code by clicking inside the cell (box with code in it) with the code you want to run and then hit CTRL + ENTER (CMD + ENTER on a Mac). You can also click in a cell and then click on the right "play" arrow to the left of the code. If you want to run all the code in your notebook, you can use the double, "fast forward" arrows at the bottom of the notebook editor.

Here's what we're going to do today:

* [Take a first look at the data](#Take-a-first-look-at-the-data)
* [See how many missing data points we have](#See-how-many-missing-data-points-we-have)
* [Figure out why the data is missing](#Figure-out-why-the-data-is-missing)
* [Drop missing values](#Drop-missing-values)
* [Filling in missing values](#Filling-in-missing-values)

Let's get started!

# Take a first look at the data
________

The first thing we'll need to do is load in the libraries and datasets we'll be using. For today, I'll be using a dataset of events that occured in American Football games for demonstration, and you'll be using a dataset of building permits issued in San Francisco.

> **Important!** Make sure you run this cell yourself or the rest of your code won't work!

In [5]:
# modules we'll use
import pandas as pd
import numpy as np

# read in all our data
df = pd.read_csv("shit.csv")

# Technique 1: The "Completely Empty" Drop
# Excel often leaves trailing rows of literal commas. This drops rows where EVERY column is NaN.
df = df.dropna(how='all')

# Reset the index so your row numbers are clean and sequential again
df = df.reset_index(drop=True)
# nfl_data = pd.read_csv("data.csv")
# sf_permits = pd.read_csv("../input/building-permit-applications-data/Building_Permits.csv")
nfl_data = df

# set seed for reproducibility
np.random.seed(0) 

The first thing I do when I get a new dataset is take a look at some of it. This lets me see that it all read in correctly and get an idea of what's going on with the data. In this case, I'm looking to see if I see any missing values, which will be reprsented with `NaN` or `None`.

In [6]:
# look at a few rows of the nfl_data file. I can see a handful of missing data already!
nfl_data.sample(5)

,Survey Year,Timestamp,SalaryUSD,Country,PostalCode,PrimaryDatabase,YearsWithThisDatabase,OtherDatabases,EmploymentStatus,JobTitle,...,HowManyCompanies,OtherPeopleOnYourTeam,DatabaseServers,PopulationOfLargestCityWithin20Miles,EmploymentSector,CareerPlansThisYear,Gender,DataAndServersAreIn,MonthsUnemployed,Counter
14530,2017,1/5/2017 11:17:51,"125,000",United States,Not Asked,Microsoft SQL Server,10,NaN,Full time employee,"Developer: Business Intelligence (SSRS, PowerB...",...,Not Asked,3,12,Not Asked,Private business,Not Asked,Not Asked,NaN,NaN,1
11745,2018,12/12/2017 11:27:40,"95,000",United States,15222,Microsoft SQL Server,16,Oracle,Full time employee,DBA (Production Focus - build & troubleshoot s...,...,Not Asked,3,200,Not Asked,Non-profit,"Stay with the same employer, same role",Male,NaN,NaN,1
3460,2023,11/22/2022 9:17:50,"100,000",United States,73003,Microsoft SQL Server,22,"MongoDB, Microsoft Access, Azure SQL DB (any f...",Full time employee,"Developer: App code (C#, JS, etc)",...,4,4,1,300K-1M (large city),Private business,"Stay with the same employer, same role",Male,NaN,NaN,1
12674,2017,1/9/2017 11:08:59,"65,000",United States,Not Asked,Microsoft SQL Server,4,"MySQL/MariaDB, Microsoft Access",Full time employee,Analyst,...,Not Asked,NaN,55,Not Asked,Private business,Not Asked,Not Asked,NaN,NaN,1
9894,2018,12/20/2017 12:03:52,"155,000",United States,92821,Microsoft SQL Server,12,"MySQL/MariaDB, Microsoft Access",Full time employee,Manager,...,Not Asked,1,8,Not Asked,Private business,"Stay with the same employer, but change roles",Male,NaN,NaN,1


Yep, it looks like there's some missing values. What about in the sf_permits dataset?

In [ ]:
# your turn! Look at a couple of rows from the sf_permits dataset. Do you notice any missing data?

# your code goes here :)

# See how many missing data points we have
___

Ok, now we know that we do have some missing values. Let's see how many we have in each column. 

In [11]:
# get the number of missing data points per column
missing_values_count = nfl_data.isnull().sum()

# look at the # of missing points in the first ten columns
missing_values_count[:]

Survey Year                                 0
Timestamp                                   0
 SalaryUSD                                  0
Country                                     0
PostalCode                               3073
PrimaryDatabase                             0
YearsWithThisDatabase                       0
OtherDatabases                           2698
EmploymentStatus                            0
JobTitle                                    0
ManageStaff                                 0
YearsWithThisTypeOfJob                      0
HowManyCompanies                            0
OtherPeopleOnYourTeam                    5550
DatabaseServers                             0
PopulationOfLargestCityWithin20Miles        0
EmploymentSector                            0
CareerPlansThisYear                         0
Gender                                      1
DataAndServersAreIn                     13854
MonthsUnemployed                        14773
Counter                           

That seems like a lot! It might be helpful to see what percentage of the values in our dataset were missing to give us a better sense of the scale of this problem:

In [13]:
# how many total missing values do we have?
total_cells = np.prod(nfl_data.shape)
total_missing = missing_values_count.sum()

# percent of data that is missing
(total_missing/total_cells) * 100

np.float64(12.212412646201065)

Wow, almost a quarter of the cells in this dataset are empty! In the next step, we're going to take a closer look at some of the columns with missing values and try to figure out what might be going on with them.

In [ ]:
# your turn! Find out what percent of the sf_permits dataset is missing


# Figure out why the data is missing
____
 
This is the point at which we get into the part of data science that I like to call "data intution", by which I mean "really looking at your data and trying to figure out why it is the way it is and how that will affect your analysis". It can be a frustrating part of data science, especially if you're newer to the field and don't have a lot of experience. For dealing with missing values, you'll need to use your intution to figure out why the value is missing. One of the most important question you can ask yourself to help figure this out is this:

> **Is this value missing becuase it wasn't recorded or becuase it dosen't exist?**

If a value is missing becuase it doens't exist (like the height of the oldest child of someone who doesn't have any children) then it doesn't make sense to try and guess what it might be. These values you probalby do want to keep as NaN. On the other hand, if a value is missing becuase it wasn't recorded, then you can try to guess what it might have been based on the other values in that column and row. (This is called "imputation" and we'll learn how to do it next! :)

Let's work through an example. Looking at the number of missing values in the nfl_data dataframe, I notice that the column `TimesSec` has a lot of missing values in it: 

In [ ]:
# look at the # of missing points in the first ten columns
gh = nfl_data["Gender"].isnull()
result_df= gh.loc[gh==True]
result_df
nfl_data.loc[4738]

4738    True
Name: Gender, dtype: bool

In [56]:
# Overwrite the original variable with the cleaned version
nfl_data = nfl_data.dropna(subset=["Gender"])

# Now this will return an empty series
gh = nfl_data["Gender"].isnull()
result_df = gh.loc[gh==True]
result_df

missing_values_count = nfl_data.isnull().sum()

# look at the # of missing points in the first ten columns
missing_values_count[:]


Survey Year                                 0
Timestamp                                   0
 SalaryUSD                                  0
Country                                     0
PostalCode                               3073
PrimaryDatabase                             0
YearsWithThisDatabase                       0
OtherDatabases                           2698
EmploymentStatus                            0
JobTitle                                    0
ManageStaff                                 0
YearsWithThisTypeOfJob                      0
HowManyCompanies                            0
OtherPeopleOnYourTeam                    5549
DatabaseServers                             0
PopulationOfLargestCityWithin20Miles        0
EmploymentSector                            0
CareerPlansThisYear                         0
Gender                                      0
DataAndServersAreIn                     13853
MonthsUnemployed                        14772
Counter                           

By looking at [the documentation](https://www.kaggle.com/maxhorowitz/nflplaybyplay2009to2016), I can see that this column has information on the number of seconds left in the game when the play was made. This means that these values are probably missing because they were not recorded, rather than because they don't exist. So, it would make sense for us to try and guess what they should be rather than just leaving them as NA's.

On the other hand, there are other fields, like `PenalizedTeam` that also have lot of missing fields. In this case, though, the field is missing because if there was no penalty then it doesn't make sense to say *which* team was penalized. For this column, it would make more sense to either leave it empty or to add a third value like "neither" and use that to replace the NA's.

> **Tip:** This is a great place to read over the dataset documentation if you haven't already! If you're working with a dataset that you've gotten from another person, you can also try reaching out to them to get more information.

If you're doing very careful data analysis, this is the point at which you'd look at each column individually to figure out the best strategy for filling those missing values. For the rest of this notebook, we'll cover some "quick and dirty" techniques that can help you with missing values but will probably also end up removing some useful information or adding some noise to your data.

## Your turn!

* Look at the columns `Street Number Suffix` and `Zipcode` from the `sf_permits` datasets. Both of these contain missing values. Which, if either, of these are missing because they don't exist? Which, if either, are missing because they weren't recorded?

# Drop missing values
___

If you're in a hurry or don't have a reason to figure out why your values are missing, one option you have is to just remove any rows or columns that contain missing values. (Note: I don't generally recommend this approch for important projects! It's usually worth it to take the time to go through your data and really look at all the columns with missing values one-by-one to really get to know your dataset.)  

If you're sure you want to drop rows with missing values, pandas does have a handy function, `dropna()` to help you do this. Let's try it out on our NFL dataset!

In [45]:
# remove all the rows that contain a missing value
nfl_data.dropna()

,Survey Year,Timestamp,SalaryUSD,Country,PostalCode,PrimaryDatabase,YearsWithThisDatabase,OtherDatabases,EmploymentStatus,JobTitle,...,HowManyCompanies,OtherPeopleOnYourTeam,DatabaseServers,PopulationOfLargestCityWithin20Miles,EmploymentSector,CareerPlansThisYear,Gender,DataAndServersAreIn,MonthsUnemployed,Counter
20,2026,1/7/2026 4:04:58,"100,000",Italy,20100,Microsoft SQL Server,12,"Microsoft SQL Server, MySQL/MariaDB, PostgreSQ...",Unemployed (and looking),DBA (General - splits time evenly between writ...,...,3,More than 5,100,300K-1M (large city),Private business,Change both employers and roles,Male,"AWS, Azure",3.0,1
32,2026,1/6/2026 3:23:45,"120,000",United States,85119,Microsoft SQL Server,25,Microsoft SQL Server,Unemployed (and looking),DBA (General - splits time evenly between writ...,...,6 or more,1,6,300K-1M (large city),Private business,"Stay with the same employer, same role",Male,"Our Own Data Centers, Rented Datacenter Space ...",15.0,1
37,2026,1/5/2026 16:49:02,"95,000",United States,37086,Microsoft SQL Server,5,Microsoft SQL Server,Unemployed (and looking),Developer: T-SQL,...,4,5,30,300K-1M (large city),Private business,"Stay with the same role, but change employers",Male,"Azure, Our Own Data Centers, Rented Datacenter...",9.0,1
77,2026,1/4/2026 17:00:12,"68,900",France,85150,Microsoft SQL Server,15,MongoDB,Unemployed (and looking),DBA (General - splits time evenly between writ...,...,2 (I worked at another similar position elsewh...,2,12,20K-99K (large town),Private business,"Stay with the same role, but change employers",Male,Azure,7.0,1
88,2026,1/2/2026 20:00:35,"150,000",United States,98027,Microsoft SQL Server,25,"Microsoft SQL Server, PostgreSQL",Unemployed (and looking),"Developer: App code (C#, JS, etc)",...,6 or more,5,3,300K-1M (large city),Private business,Prefer not to say,Female,AWS,16.0,1
129,2026,1/1/2026 9:30:19,"98,000",United Kingdom,LS,Microsoft SQL Server,15,"PostgreSQL, Azure SQL DB (any flavor)",Full time employee,Developer: T-SQL,...,3,1,40,1M+ (metropolis),Private business,"Stay with the same role, but change employers",Male,"AWS, Azure, Our Own Data Centers",30.0,1
155,2026,12/31/2025 16:45:18,"110,000",United States,554,Microsoft SQL Server,27,"Azure SQL DB (any flavor), Sybase Adaptive Ser...",Unemployed (and looking),Developer: T-SQL,...,6 or more,4,5,300K-1M (large city),Private business,"Stay with the same role, but change employers",Male,"Azure, Our Own Data Centers",9.0,1
157,2026,12/31/2025 16:35:44,"120,000",United States,59912,Microsoft SQL Server,25,Microsoft SQL Server,Full time employee,DBA (General - splits time evenly between writ...,...,3,1,50,20K-99K (large town),Private business,"Stay with the same employer, same role",Male,"Azure, Our Own Data Centers",20.0,1
162,2026,12/31/2025 15:02:23,"140,000",United States,89052,Microsoft SQL Server,30,Oracle,Full time employee,DBA (General - splits time evenly between writ...,...,3,3,100,1M+ (metropolis),Local government,"Stay with the same employer, same role",Male,Our Own Data Centers,0.0,1
163,2026,12/31/2025 14:58:43,"86,000",United States,63139,Microsoft SQL Server,8,"Microsoft SQL Server, Oracle, Azure SQL DB (an...",Full time employee,DBA (Production Focus - build & troubleshoot s...,...,3,3,32,300K-1M (large city),Non-profit,"Stay with the same employer, same role",Female,Azure,42.0,1


Oh dear, it looks like that's removed all our data! 😱 This is because every row in our dataset had at least one missing value. We might have better luck removing all the *columns* that have at least one missing value instead.

In [46]:
# remove all columns with at least one missing value
columns_with_na_dropped = nfl_data.dropna(axis=1)
columns_with_na_dropped.head()

,Survey Year,Timestamp,SalaryUSD,Country,PrimaryDatabase,YearsWithThisDatabase,EmploymentStatus,JobTitle,ManageStaff,YearsWithThisTypeOfJob,HowManyCompanies,DatabaseServers,PopulationOfLargestCityWithin20Miles,EmploymentSector,CareerPlansThisYear,Counter
0,2026,1/10/2026 11:03:31,"132,000",United States,Azure SQL DB,7,Full time employee,"Developer: App code (C#, JS, etc)",No,16,1 (this is the only company where I've had thi...,40,300K-1M (large city),Private business,"Stay with the same employer, same role",1
1,2026,1/9/2026 17:39:36,"125,000",United States,Microsoft SQL Server,25,Full time employee,Engineer,No,28,6 or more,5,100K-299K (city),Private business,"Stay with the same employer, same role",1
2,2026,1/9/2026 16:00:31,"110,000",United States,Microsoft SQL Server,10,Full time employee,Developer: T-SQL,No,4,1 (this is the only company where I've had thi...,5,1M+ (metropolis),Private business,"Stay with the same employer, but change roles",1
3,2026,1/9/2026 14:42:23,"130,000",United States,Microsoft SQL Server,11,Full time employee,Developer: T-SQL,No,11,2 (I worked at another similar position elsewh...,7,1M+ (metropolis),Private business,"Stay with the same employer, but change roles",1
4,2026,1/9/2026 10:16:12,"176,000",United States,Microsoft SQL Server,20,Full time employee,Engineer,No,20,2 (I worked at another similar position elsewh...,17,1M+ (metropolis),Private business,"Stay with the same employer, same role",1


In [47]:
# just how much data did we lose?
print("Columns in original dataset: %d \n" % nfl_data.shape[1])
print("Columns with na's dropped: %d" % columns_with_na_dropped.shape[1])

Columns in original dataset: 22 

Columns with na's dropped: 16


We've lost quite a bit of data, but at this point we have successfully removed all the `NaN`'s from our data. 

In [48]:
# Your turn! Try removing all the rows from the sf_permits dataset that contain missing values. How many are left?


In [49]:
# Now try removing all the columns with empty values. Now how much of your data is left?


# Filling in missing values automatically
_____

Another option is to try and fill in the missing values. For this next bit, I'm getting a small sub-section of the NFL data so that it will print well.

In [50]:
# get a small subset of the NFL dataset
subset_nfl_data = nfl_data.loc[:, 'EPA':'Season'].head()
subset_nfl_data

KeyError: 'EPA'

We can use the Panda's fillna() function to fill in missing values in a dataframe for us. One option we have is to specify what we want the `NaN` values to be replaced with. Here, I'm saying that I would like to replace all the `NaN` values with 0.

In [ ]:
# replace all NA's with 0
subset_nfl_data.fillna(0)

I could also be a bit more savvy and replace missing values with whatever value comes directly after it in the same column. (This makes a lot of sense for datasets where the observations have some sort of logical order to them.)

In [ ]:
# replace all NA's the value that comes directly after it in the same column, 
# then replace all the reamining na's with 0
subset_nfl_data.fillna(method = 'bfill', axis=0).fillna(0)

Filling in missing values is also known as "imputation", and you can find more exercises on it [in this lesson, also linked under the "More practice!" section](https://www.kaggle.com/dansbecker/handling-missing-values). First, however, why don't you try replacing some of the missing values in the sf_permit dataset?

In [ ]:
# Your turn! Try replacing all the NaN's in the sf_permits data with the one that
# comes directly after it and then replacing any remaining NaN's with 0

And that's it for today! If you have any questions, be sure to post them in the comments below or [on the forums](https://www.kaggle.com/questions-and-answers). 

Remember that your notebook is private by default, and in order to share it with other people or ask for help with it, you'll need to make it public. First, you'll need to save a version of your notebook that shows your current work by hitting the "Commit & Run" button. (Your work is saved automatically, but versioning your work lets you go back and look at what it was like at the point you saved it. It also let's you share a nice compiled notebook instead of just the raw code.) Then, once your notebook is finished running, you can go to the Settings tab in the panel to the left (you may have to expand it by hitting the [<] button next to the "Commit & Run" button) and setting the "Visibility" dropdown to "Public".

# More practice!
___

If you're looking for more practice handling missing values, check out these extra-credit\* exercises:

* [Handling Missing Values](https://www.kaggle.com/dansbecker/handling-missing-values): In this notebook Dan shows you several approaches to imputing missing data using scikit-learn's imputer. 
* Look back at the `Zipcode` column in the `sf_permits` dataset, which has some missing values. How would you go about figuring out what the actual zipcode of each address should be? (You might try using another dataset. You can search for datasets about San Fransisco on the [Datasets listing](https://www.kaggle.com/datasets).) 

\* no actual credit is given for completing the challenge, you just learn how to clean data real good :P